In [1]:
import xgboost as xgb
import pandas as pd
import src.train_utils as T
from sklearn.dummy import DummyRegressor

In [14]:
df = pd.read_csv('../datasets/exp_time_lags_dataset.csv')
df['date'] = pd.to_datetime(df['date'])

train_df = df[df['date'].dt.year < 2022]

In [15]:
train_df['day_of_week'] = train_df['date'].dt.dayofweek
dow_dummies = pd.get_dummies(train_df['day_of_week'], prefix='dow')
train_df = pd.concat([train_df, dow_dummies], axis=1)

train_df['month'] = train_df['date'].dt.month
month_dummies = pd.get_dummies(train_df['month'], prefix='month')
train_df = pd.concat([train_df, month_dummies], axis=1)


train_df['is_weekend'] = train_df['date'].dt.dayofweek >= 5
train_df['is_weekend'] = train_df['is_weekend'].astype(int)


In [19]:
train_df['date'] = pd.to_datetime(train_df['date'])

In [25]:
base = []

feature_experiments = [
    ('persistence', base),
]

model=DummyRegressor(strategy='constant', constant=0)

results = T.run_experiments(
    df=train_df, 
    model=model, 
    feature_experiments=feature_experiments, 
    train_days=365*2,
    gap_days=21,
    val_days=49
)
print(results.sort_values('weighted_rmse'))

Running experiment: persistence
Fold 0
RMSE: 1.7836516750347196
Fold 1
RMSE: 6.286165987056864
Fold 2
RMSE: 5.223866135065733
Fold 3
RMSE: 9.354228728946818
Fold 4
RMSE: 14.504135042313575
Fold 5
RMSE: 2.214256839087764
Fold 6
RMSE: 1.003455618636538
Fold 7
RMSE: 1.0617824382856687
Fold 8
RMSE: 1.9852754348402382
Fold 9
RMSE: 4.290847572626642
    experiment  n_features  unweighted_rmse  weighted_rmse
0  persistence           0         4.770767       6.657028


In [22]:
base = ['row', 'col', 'pm25_t', 'u_wind_10m_t', 'v_wind_10m_t',
       'dew_temp_2m_t', 'temp_2m_t', 'surf_pressure_t', 'precip_sum_t',
       'frp_t', 'elevation_t', 'delta_pm25_t',
       'frp_t_mean_3x3', 'frp_t_mean_5x5', 'frp_t_mean_7x7',
       'delta_pm25_t-lag1', 'delta_pm25_t-lag2', 'delta_pm25_t_ma3',
       'delta_pm25_t_ma5']


params = {
    'max_depth': 3,            
    'learning_rate': 0.1,     
    'n_estimators': 45,     
    'subsample': 0.8,          
    'colsample_bytree': 0.8,   
}

feature_experiments = [
    ('base', base),
    ('base + is_weekend', base + ['is_weekend']),
    ('base + day_of_week', base + ['day_of_week']),
    ('base + dow_one_hot', base + ['dow_0', 'dow_1', 'dow_2', 'dow_3', 'dow_4', 'dow_5', 'dow_6']),
    ('base + month', base + ['month']),
    ('base + month_one_hot', base + ['month_1', 'month_2', 'month_3', 'month_4', 'month_5', 'month_6', 'month_7', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12'])
]

model=xgb.XGBRegressor(**params, random_state=191)

results = T.run_experiments(
    df=train_df, 
    model=model, 
    feature_experiments=feature_experiments, 
    train_days=365*2,
    gap_days=21,
    val_days=49
)

print(results.sort_values('weighted_rmse'))

Running experiment: base
Fold 0
RMSE: 1.7019354118601202
Fold 1
RMSE: 5.396397693660629
Fold 2
RMSE: 5.144846264882276
Fold 3
RMSE: 8.625408372329169
Fold 4
RMSE: 13.319751184712494
Fold 5
RMSE: 2.136991714619597
Fold 6
RMSE: 0.969587268318953
Fold 7
RMSE: 1.0416339106709342
Fold 8
RMSE: 1.9029071357009848
Fold 9
RMSE: 4.073211358538501
Running experiment: base + is_weekend
Fold 0
RMSE: 1.7135546020994643
Fold 1
RMSE: 5.475148700356554
Fold 2
RMSE: 5.114430779284156
Fold 3
RMSE: 8.625663518966663
Fold 4
RMSE: 13.247694655712541
Fold 5
RMSE: 2.1747536644085947
Fold 6
RMSE: 0.9775308803486847
Fold 7
RMSE: 1.049032872369633
Fold 8
RMSE: 1.9023609054785215
Fold 9
RMSE: 4.083310794685574
Running experiment: base + day_of_week
Fold 0
RMSE: 1.7085899148954238
Fold 1
RMSE: 5.518676630125903
Fold 2
RMSE: 5.105682484285207
Fold 3
RMSE: 8.610934933259415
Fold 4
RMSE: 13.196097959175074
Fold 5
RMSE: 2.162369231445931
Fold 6
RMSE: 0.964933994836693
Fold 7
RMSE: 1.051551366739552
Fold 8
RMSE: 1.9098

In [24]:
base = ['row', 'col', 'pm25_t', 'u_wind_10m_t', 'v_wind_10m_t',
       'dew_temp_2m_t', 'temp_2m_t', 'surf_pressure_t', 'precip_sum_t',
       'frp_t', 'elevation_t', 'delta_pm25_t',
       'frp_t_mean_3x3', 'frp_t_mean_5x5', 'frp_t_mean_7x7',
       'delta_pm25_t-lag1', 'delta_pm25_t-lag2', 'delta_pm25_t_ma3',
       'delta_pm25_t_ma5']


params = {
    'max_depth': 3,            
    'learning_rate': 0.1,     
    'n_estimators': 45,     
    'subsample': 0.8,          
    'colsample_bytree': 0.8,
    'reg_alpha': 1,
    'reg_lambda': 10,   
}

feature_experiments = [
    ('base', base),
    ('base + is_weekend', base + ['is_weekend']),
    ('base + day_of_week', base + ['day_of_week']),
    ('base + dow_one_hot', base + ['dow_0', 'dow_1', 'dow_2', 'dow_3', 'dow_4', 'dow_5', 'dow_6']),
    ('base + month', base + ['month']),
    ('base + month_one_hot', base + ['month_1', 'month_2', 'month_3', 'month_4', 'month_5', 'month_6', 'month_7', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12'])
]

model=xgb.XGBRegressor(**params, random_state=191)

results = T.run_experiments(
    df=train_df, 
    model=model, 
    feature_experiments=feature_experiments, 
    train_days=365*2,
    gap_days=21,
    val_days=49
)

print(results.sort_values('weighted_rmse'))

Running experiment: base
Fold 0
RMSE: 1.7007120875300095
Fold 1
RMSE: 5.371716130754097
Fold 2
RMSE: 5.140885289350643
Fold 3
RMSE: 8.639757888207185
Fold 4
RMSE: 13.243259116366357
Fold 5
RMSE: 2.149329664673667
Fold 6
RMSE: 0.9705722172598049
Fold 7
RMSE: 1.0370656339397057
Fold 8
RMSE: 1.8926544116442232
Fold 9
RMSE: 4.080291865304567
Running experiment: base + is_weekend
Fold 0
RMSE: 1.714010033723113
Fold 1
RMSE: 5.476296429363312
Fold 2
RMSE: 5.094868777193189
Fold 3
RMSE: 8.618786482769192
Fold 4
RMSE: 13.258226244744357
Fold 5
RMSE: 2.1514691102466594
Fold 6
RMSE: 0.9757467300940464
Fold 7
RMSE: 1.0354855056021843
Fold 8
RMSE: 1.904940330381014
Fold 9
RMSE: 4.099020754974593
Running experiment: base + day_of_week
Fold 0
RMSE: 1.7103407535475152
Fold 1
RMSE: 5.517076168725834
Fold 2
RMSE: 5.096007132995855
Fold 3
RMSE: 8.62928250787076
Fold 4
RMSE: 13.298238245954682
Fold 5
RMSE: 2.1755723389852606
Fold 6
RMSE: 0.9718480980853056
Fold 7
RMSE: 1.0429455145443352
Fold 8
RMSE: 1.90

In [27]:
base = ['row', 'col', 'pm25_t', 'u_wind_10m_t', 'v_wind_10m_t',
       'dew_temp_2m_t', 'temp_2m_t', 'surf_pressure_t', 'precip_sum_t',
       'frp_t', 'elevation_t', 'delta_pm25_t',
       'frp_t_mean_3x3', 'frp_t_mean_5x5', 'frp_t_mean_7x7',
       'delta_pm25_t-lag1', 'delta_pm25_t-lag2', 'delta_pm25_t_ma3',
       'delta_pm25_t_ma5']


params = {
    'max_depth': 3,            
    'learning_rate': 0.1,     
    'n_estimators': 45,     
    'subsample': 0.8,          
    'colsample_bytree': 0.8,   
}

feature_experiments = [
    ('base', base),
    ('base + is_weekend', base + ['is_weekend']),
    ('base + day_of_week', base + ['day_of_week']),
    ('base + month', base + ['month']),
    ('base + is_weekend + day_of_week', base + ['is_weekend', 'day_of_week']),
    ('base + is_weekend + month', base + ['is_weekend', 'month']),
    ('base + day_of_week + month', base + ['day_of_week', 'month']),
    ('base + is_weekend + day_of_week + month', base + ['is_weekend', 'day_of_week', 'month'])
]

model=xgb.XGBRegressor(**params, random_state=191)

results = T.run_experiments(
    df=train_df, 
    model=model, 
    feature_experiments=feature_experiments, 
    train_days=365*2,
    gap_days=21,
    val_days=49
)

print(results.sort_values('weighted_rmse'))

Running experiment: base
Fold 0
RMSE: 1.7019354118601202
Fold 1
RMSE: 5.396397693660629
Fold 2
RMSE: 5.144846264882276
Fold 3
RMSE: 8.625408372329169
Fold 4
RMSE: 13.319751184712494
Fold 5
RMSE: 2.136991714619597
Fold 6
RMSE: 0.969587268318953
Fold 7
RMSE: 1.0416339106709342
Fold 8
RMSE: 1.9029071357009848
Fold 9
RMSE: 4.073211358538501
Running experiment: base + is_weekend
Fold 0
RMSE: 1.7135546020994643
Fold 1
RMSE: 5.475148700356554
Fold 2
RMSE: 5.114430779284156
Fold 3
RMSE: 8.625663518966663
Fold 4
RMSE: 13.247694655712541
Fold 5
RMSE: 2.1747536644085947
Fold 6
RMSE: 0.9775308803486847
Fold 7
RMSE: 1.049032872369633
Fold 8
RMSE: 1.9023609054785215
Fold 9
RMSE: 4.083310794685574
Running experiment: base + day_of_week
Fold 0
RMSE: 1.7085899148954238
Fold 1
RMSE: 5.518676630125903
Fold 2
RMSE: 5.105682484285207
Fold 3
RMSE: 8.610934933259415
Fold 4
RMSE: 13.196097959175074
Fold 5
RMSE: 2.162369231445931
Fold 6
RMSE: 0.964933994836693
Fold 7
RMSE: 1.051551366739552
Fold 8
RMSE: 1.9098

In [28]:
train_df.columns

Index(['row', 'col', 'date', 'pm25_t', 'u_wind_10m_t', 'v_wind_10m_t',
       'dew_temp_2m_t', 'temp_2m_t', 'surf_pressure_t', 'precip_sum_t',
       'frp_t', 'elevation_t', 'delta_pm25_t+1', 'delta_pm25_t',
       'frp_t_mean_3x3', 'frp_t_mean_5x5', 'frp_t_mean_7x7',
       'delta_pm25_t-lag1', 'delta_pm25_t-lag2', 'delta_pm25_t_ma3',
       'delta_pm25_t_ma5', 'day_of_week', 'dow_0', 'dow_1', 'dow_2', 'dow_3',
       'dow_4', 'dow_5', 'dow_6', 'month', 'month_1', 'month_2', 'month_3',
       'month_4', 'month_5', 'month_6', 'month_7', 'month_8', 'month_9',
       'month_10', 'month_11', 'month_12', 'is_weekend'],
      dtype='object')

In [29]:
exp_df = train_df.drop(['day_of_week', 'dow_0', 'dow_1', 'dow_2', 'dow_3',
       'dow_4', 'dow_5', 'dow_6', 'month_1', 'month_2', 'month_3',
       'month_4', 'month_5', 'month_6', 'month_7', 'month_8', 'month_9',
       'month_10', 'month_11', 'month_12', 'is_weekend'], axis=1)

In [31]:
exp_df.to_csv('exp_seasonality_dataset.csv', index=False)